# Data + Logistic Regression

Owner: Madushanka D.M.C  (proposal sections 4.2 to 4.5, 4.8, 4.9)

Put `PhiUSIIL_Phishing_URL_Dataset.csv` in `data/raw/` before running. Run from `notebooks/`.

Target convention: `data.get_target()` gives **1 = phishing**, 0 = legitimate
(raw `label` 0 = phishing). Precision, recall and ROC-AUC all refer to phishing.

## To do

- [x] Inspect the dataset: columns, data types, target distribution (4.2)
- [x] Check the available files and combine only compatible ones (4.3): one file only
- [x] Find and handle duplicates, missing and inconsistent values (4.4)
- [x] Group decision (2026-11-24): near-duplicate URLs form URL groups; group-aware split and grouped CV (section 4b)
- [x] Group decision: `URLSimilarityIndex` dropped from every feature set (section 7)
- [x] Encoding / scaling decisions, consistent train/test split (4.5, 4.8)
- [x] Move the final cleaning and grouping code into `src/data.py`
- [x] Retune Logistic Regression with grouped 5-fold CV on the training split, `url_engineered` (section 6)
- [x] Document the URL-format collection bias with a reference rule (section 7b)
- [x] Final test-set report: produced once by `python main.py` (section 8 displays it)

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src import config, data, evaluate, features
from src.models import logreg

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

PRIMARY = features.PRIMARY_FEATURE_SET   # "url_engineered"

# Quick checks only (section 5). Inspection, grouping and the LR evidence use the
# full data, whatever SAMPLE is (the tuning tables are read from reports/tuning/).
SAMPLE = 20000

# Keep False until tuning and the group decisions are final. The test set is
# for the final report only, never for choosing hyperparameters or features.
RUN_FINAL_TEST = False

## 2. Dataset inspection (4.2, 4.3)

There is a single CSV (`config.RAW_FILE`), so there are no files to merge (4.3).

In [ ]:
raw = data.load_raw()
raw_snapshot = raw.copy(deep=True)   # used below to prove clean() does not modify its input
print("shape:", raw.shape)
print("dtypes:", raw.dtypes.astype(str).value_counts().to_dict())
print("missing values (all columns):", int(raw.isna().sum().sum()))
print("missing target:", int(raw[config.TARGET].isna().sum()))
print("raw label counts (1 = legitimate, 0 = phishing):", raw[config.TARGET].value_counts().to_dict())
print("fully identical rows:", int(raw.duplicated().sum()))
print("rows whose URL already appeared earlier:", int(raw["URL"].duplicated().sum()))

## 3. Exact duplicate URLs (4.4)

`clean()` keeps the **first occurrence** of each repeated URL (file order), so
the same URL can never be in both train and test. The table below checks what
that rule throws away.

In [ ]:
dups = data.duplicate_url_report(raw)
print("repeated URLs:", len(dups))
print("group sizes:", dups["n_rows"].value_counts().to_dict())
print("URLs whose copies have different labels:", int((dups["n_labels"] > 1).sum()))
print("URLs whose copies have different feature values:", int((dups["n_features_differing"] > 0).sum()))

# How often each feature differs between the copies of the same URL.
dups["features_differing"].explode().value_counts().head(15)

In [ ]:
# One example: the URL string is identical, but URLLength and other features are not.
example_url = dups.loc[dups["n_features_differing"].idxmax(), "URL"]
cols = ["URL", config.TARGET] + dups.loc[dups["URL"] == example_url, "features_differing"].iloc[0][:8]
raw.loc[raw["URL"] == example_url, cols]

**Decision (approved): keep the first occurrence; record that the copies differ.**

- *What:* for each repeated URL, the first row in file order is kept and the later row is dropped.
- *Why:* the copies never disagree on the label, so no label information is lost.
  Keeping one copy prevents the same URL from being in both train and test,
  which would inflate the scores. "Keep first" is deterministic, so every member gets the same rows.
- *Caveat:* the copies are **not identical records**. Their feature values differ
  (see the counts above; even `URLLength` differs for the same URL string), so the
  dataset's features were probably computed in separate crawls or by different
  versions of the feature extractor. The kept row is a convention, not the "correct" one.
- *Alternatives:* averaging the copies would invent feature values that were never
  observed. Dropping both copies loses a correctly labelled URL. Keeping both copies
  risks them being split across train and test. "Keep first" is the simplest rule that avoids all three.

In [ ]:
df = data.clean(raw)
pd.testing.assert_frame_equal(raw, raw_snapshot)   # clean() returned a copy; raw is unchanged
data.check_target(df)                              # labels are only 0 and 1 after cleaning
print("rows: raw", len(raw), "-> clean", len(df), f"(removed {len(raw) - len(df)})")
print("target after cleaning (1 = phishing):", data.get_target(df).value_counts().to_dict())

*Findings, full data, run of 2026-11-17 (outputs are cleared on save; rerun to regenerate):*
the file has 235,795 rows, no missing values and no fully identical rows. 425 URLs appear exactly
twice; no pair disagrees on the label, but **all 425 pairs differ in at least one feature value**
(`URLLength` in every pair). All 425 removed rows are phishing, so cleaning leaves 235,370 rows
(100,520 phishing, 134,850 legitimate). Labels are only 0/1 after cleaning (`check_target` passes).

## 4. Near-duplicate URLs: first report (2026-11-17, broader key)

A near-duplicate pair is two **different** URL strings that become equal after lower-casing and removing
trailing `/`. This was the first report for the group. **Superseded by the group decision in section 4b**,
which uses `data.url_group_key()` (path and query case kept) and keeps whole groups on one side of every split.

In [ ]:
pairs = data.near_duplicate_url_pairs(df)
summary = {
    "near-duplicate pairs": len(pairs),
    "pairs with conflicting labels": int(pairs["labels_conflict"].sum()),
    "differ by letter case": int(pairs["differs_by_case"].sum()),
    "differ by trailing slash(es)": int(pairs["differs_by_trailing_slash"].sum()),
    "root path only (e.g. https://a.com vs https://a.com/)": int(pairs["root_path_only"].sum()),
    "with a non-empty path, query or fragment": int((~pairs["root_path_only"]).sum()),
}
pd.Series(summary, name="count")

In [ ]:
# The pair(s) with conflicting labels (raw label: 1 = legitimate, 0 = phishing).
conflict = pairs[pairs["labels_conflict"]]
conflict_rows = pd.concat([conflict["row_a"], conflict["row_b"]])
df.loc[conflict_rows, ["URL", "Domain", config.TARGET, "URLSimilarityIndex", "URLLength"]]

In [ ]:
# Would the shared split put the two halves of a pair on different sides?
# Only row indices are used here; no test labels or scores are looked at.
X_tr_idx, X_te_idx, _, _ = data.split(df, data.get_original_features(df))
in_test = df.index.isin(X_te_idx.index)
crosses = in_test[pairs["row_a"]] != in_test[pairs["row_b"]]
print("pairs split across train and test:", int(crosses.sum()), "of", len(pairs))
pairs.loc[~pairs["root_path_only"], ["url_a", "url_b", "label_a", "label_b"]].head(8)

*Findings, run of 2026-11-17 (historical):* 1,165 near-duplicate pairs. **None differ by letter case; all
differ only in trailing slashes** (one pair, `https://grin-6cbd8.web.app/#////` vs `.../#///`, differs in
the number of slashes inside a fragment). 461 pairs are root-only (`https://a.com` vs `https://a.com/`),
704 have a path, query or fragment. **One pair has conflicting labels:** `https://www.education.gouv.fr/`
is labelled phishing and `https://www.education.gouv.fr` legitimate. Under the **old random split**, the two
halves of 392 of the 1,165 pairs fell on different sides of train/test. With the grouped split (section 4b) the
cell above reports **0 of 1,165** (full-data run of 2026-11-24).

**Why case changes and trailing slashes are not automatically "the same URL"**

- *Case:* in a URL, the scheme and host are case-insensitive (`HTTP://Example.COM` = `http://example.com`),
  but the **path and query are case-sensitive** (RFC 3986). `/Login` and `/login` can be different pages on
  a case-sensitive web server (most Linux servers). Lower-casing the whole string would merge them.
  (The case count above shows whether this actually happens in PhiUSIIL.)
- *Trailing slash:* `https://a.com/path` and `https://a.com/path/` are different URIs. A server may
  redirect one to the other, serve different content, or return 404 for one of them. Only for an empty
  path (`https://a.com` vs `https://a.com/`) does RFC 3986 normalisation treat them as equivalent.
  Slashes after `#` are a fragment and never reach the server at all.
- *Model inputs differ:* many PhiUSIIL features are computed from the URL string (`URLLength`,
  character counts and ratios, `URLCharProb`, ...), so the two rows are different inputs, not copies.
  Merging needs a rule for which feature values to keep.
- *Phishing behaviour:* attackers deliberately reuse a URL with small variations, so variants may be
  genuinely separate submissions.
- *Label conflict:* the conflicting pair above is the same root URL labelled both ways. One of the two
  labels must be wrong; a blind normalise-and-drop rule would keep whichever came first.

**Counter-argument (leakage):** near-identical URLs on both sides of the split let a model partly
memorise instead of generalise, which can inflate test scores (see the "split across train and test" count).

**Options for the group:** (a) keep as is (current); (b) normalise only the safe cases
(host case, empty path `x` vs `x/`) and deduplicate; (c) keep the rows but split by normalised URL
(group-aware split) so variants stay on the same side; (d) drop the conflicting pair only.
Each option changes `clean()` or `split()`, so it must be agreed and then applied to everyone.


## 4b. Agreed decision (2026-11-24): URL groups and the group-aware split

`data.url_group_key()` normalizes a URL **only for grouping**; the `URL` column and all features keep the original string.

| Step | Example |
|---|---|
| Remove `http://` / `https://` (any case) | `HTTPS://a.com/x` → `a.com/x` |
| Lower-case the hostname | `A.COM/x` → `a.com/x` |
| Remove one leading `www.` | `www.a.com` → `a.com` |
| Remove trailing `/` from the path | `a.com/x/` → `a.com/x` |
| **Kept as written** | path case, query (case and content), fragment, port, user info |

- `data.split_index()` = first fold of `StratifiedGroupKFold(5, shuffle=True, random_state=42)` → about 20 % test, whole groups only, class balance kept.
- `evaluate.cv_folds(X, y, groups)` = the same grouped, stratified, seeded 5 folds for every model on the training rows.
- **Label-conflict groups are kept unchanged** (dataset label noise): no label is edited or row dropped; a group never crosses the split, so the conflict cannot leak.
- Why not stricter (drop the query, lower-case everything)? It merged unrelated URLs into groups of up to 45 rows.


In [ ]:

examples = ["HTTPS://WWW.Example.com/Login/?a=B", "http://example.com/Login?a=B", "https://www.example.com/login?a=B"]
display(pd.DataFrame({"URL": examples, "url_group_key": [data.url_group_key(u) for u in examples]}))

groups = data.group_ids(df)
sizes = groups.value_counts()
conflicts = data.conflicting_label_groups(df)
train_idx, test_idx = data.split_index(df)            # about 70 s on the full data (cached afterwards)
y_all = data.get_target(df)
shared = set(groups.loc[train_idx]) & set(groups.loc[test_idx])
assert not shared, "a URL group crosses the train/test split"
display(pd.Series({
    "rows": len(df), "URL groups": groups.nunique(),
    "groups with more than one row": int((sizes > 1).sum()),
    "rows in such groups": int(sizes[sizes > 1].sum()), "largest group": int(sizes.max()),
    "label-conflict groups (kept)": conflicts["url_group_key"].nunique(), "rows in them": len(conflicts),
    "train rows": len(train_idx), "test rows": len(test_idx),
    "test share": round(len(test_idx) / len(df), 4),
    "phishing share train": round(y_all.loc[train_idx].mean(), 4),
    "phishing share test": round(y_all.loc[test_idx].mean(), 4),
    "groups in both train and test": len(shared),
}).to_frame("value"))
conflicts.head(6)


## 5. Quick baseline check on a sample

A fast smoke test on `SAMPLE` rows with the old untuned baseline (lbfgs, L2, C = 1) on the primary
feature set, with grouped CV. Not used for any decision.

In [ ]:
def baseline_build():
    """The untuned starting point (previous logreg.build()), kept for comparison."""
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=config.RANDOM_STATE),
    )

df_small = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True) if SAMPLE and SAMPLE < len(df) else df
df_small = features.add_features(df_small)
tr_small, _ = data.split_index(df_small)
cols_small = features.build_feature_sets(df_small.loc[tr_small])[PRIMARY]
pd.Series(evaluate.cv_evaluate(baseline_build(), df_small.loc[tr_small, cols_small],
                               data.get_target(df_small).loc[tr_small],
                               groups=data.group_ids(df_small).loc[tr_small])).round(4)

## 6. Hyperparameter tuning (4.9), full training split, grouped CV

- **Data:** the training rows of the group-aware split (188,297 rows on the full data); the test rows are never used.
- **Features:** the primary set `url_engineered` (23 URL-only features; no `URLSimilarityIndex`, no `IsHTTPS`, no page content).
- **CV:** `evaluate.cv_folds(X, y, groups)`: the same 5 grouped, stratified, seeded folds as the Decision Tree and Random Forest.
- **Selection:** phishing F1; a candidate is **valid** only if it converged in all 5 folds (no `ConvergenceWarning`,
  `n_iter < max_iter = 1000`). Rule fixed before the run (one-standard-error): best = highest mean F1 among valid
  candidates; chosen = the **smallest C** (strongest regularisation) with mean F1 ≥ best − SE(best).
- **What happened (`python tune_models.py --parts lr` / `lr_followup`):**
  1. The full grid (L2/lbfgs C 0.001–10 and L1/liblinear C 0.001–1, 45 fits) was **stopped after 36 fits**:
     L1/liblinear C = 0.1 took 33 minutes for one fold and did **not converge**, and the remaining 9 L1 fits did
     not finish in a further 55 minutes. Its log is kept (`reports/tuning/lr_full_run_INCOMPLETE_2026-11-24.log`);
     no result of that run is used for selection, and L1 was not re-run.
  2. A bounded follow-up re-ran **L2/lbfgs only** (2 workers, time budget). Round 1 (C 0.001–100) put the best at the
     upper edge (C = 100), so one pre-registered edge check added C = 300 and 1000 (round 2, re-run in full; it
     reproduced every round-1 fold score exactly). The search stopped after round 2.

The cells below read the stored full-data tables; they do not rerun the search (about 3 minutes plus data loading).

In [ ]:
original = data.get_original_features(df)
X_train, X_test, y_train, y_test = data.split(df, original)    # grouped split; X_test is not used for any choice
train_groups = groups.loc[X_train.index]
df_feat = features.add_features(df)
sets = features.build_feature_sets(df_feat.loc[X_train.index])   # training rows only
Xp_train = df_feat.loc[X_train.index, sets[PRIMARY]]
print(f"{PRIMARY}: {len(sets[PRIMARY])} features | train rows: {len(X_train):,} | test rows (untouched): {len(X_test):,}")

tuning = config.RESULTS_DIR / "tuning"
lr_table = pd.read_csv(tuning / "lr_followup_search.csv")
show = ["penalty", "solver", "C", "valid", "n_converged", "n_iter_max", "f1_mean", "f1_std", "f1_se",
        "roc_auc_mean", "precision_mean", "recall_mean", "within_1se_of_best", "chosen"]
display(lr_table[show].round(5))

# The stopped full-grid run: what finished, per candidate (parsed from its log).
import re
log = (tuning / "lr_full_run_INCOMPLETE_2026-11-24.log").read_text()
fits = pd.DataFrame(re.findall(r"\] (l[12])/(\w+)\s+C=([\d.]+)\s+fold (\d) \| f1 ([\d.]+) n_iter\s+(\d+) (converged|NOT CONVERGED)", log),
                    columns=["penalty", "solver", "C", "fold", "f1", "n_iter", "status"])
display(fits.groupby(["penalty", "solver", "C"]).agg(folds_finished=("fold", "size"),
        not_converged=("status", lambda s: int((s != "converged").sum())), f1_mean_finished=("f1", lambda s: s.astype(float).mean())).round(5))

In [ ]:
# build() must contain the values the follow-up search chose (fixed values keep main.py fast).
chosen = lr_table.loc[lr_table["chosen"]].iloc[0]
print("one-SE choice:", chosen[["penalty", "solver", "C", "f1_mean", "f1_std"]].to_dict())
print("logreg.build():", {k: getattr(logreg.build()[-1], k) for k in ("C", "l1_ratio", "solver", "max_iter")})
assert (logreg.BEST_C, logreg.BEST_SOLVER, logreg.BEST_L1_RATIO) == (chosen["C"], chosen["solver"], 0.0), \
    "Update BEST_C / BEST_SOLVER in src/models/logreg.py"
print("C at the upper edge of the searched range:", chosen["C"] == lr_table["C"].max())

In [ ]:
# Tuned build() vs the untuned baseline, same grouped folds, training split only (about 2 minutes).
cv_compare = pd.DataFrame({
    "baseline (lbfgs, L2, C=1)": evaluate.cv_evaluate(baseline_build(), Xp_train, y_train, groups=train_groups, n_jobs=2),
    f"tuned logreg.build() (C={logreg.BEST_C:g})": evaluate.cv_evaluate(logreg.build(), Xp_train, y_train, groups=train_groups, n_jobs=2),
})
cv_compare.round(5)

*Findings, grouped protocol (2026-11-24, full training split, `url_engineered`):* all 50 follow-up fits converged.
Mean CV F1 rises steadily with C: 0.826 (C = 0.001), 0.970 (C = 1), 0.9952 (C = 10), 0.9965 (C = 100),
0.99674 ± 0.00026 (C = 1000). The one-SE rule picks **C = 1000** (threshold 0.99661; C = 300 misses by 0.00003).

- **Limitation:** C = 1000 is the upper edge of the searched range. From C = 30 on, each step adds only about one SE,
  so the curve has flattened, but the true optimum may lie at even weaker regularisation (C → ∞ is plain
  maximum-likelihood LR).
- **Why weak regularisation wins here:** the URL features almost separate the classes because every legitimate
  PhiUSIIL URL is a bare `https://www.<domain>` homepage (section 7b). A linear model needs large weights to draw that
  boundary sharply, and a strong penalty (small C) prevents it. So the high LR score is largely the collection
  format, not a general phishing signal.
- L1 (sparse) models were not evaluated to completion: liblinear did not converge at C = 0.1 within 1,000 iterations.

*Historical (2026-11-17, ungrouped random split, 50 original features incl. `URLSimilarityIndex` and page content):*
best = L2, C = 10 (liblinear), mean CV F1 0.99991; untuned baseline 0.99986. Superseded: that protocol leaked
near-duplicate URLs and a label proxy.

## 7. `URLSimilarityIndex`: evidence for the group (training split only)

**Decision taken (2026-11-19, confirmed 2026-11-24): dropped from every feature set** (`features.EXCLUDED_FEATURES`).
This section keeps the evidence. Checks, on the grouped training split:
1. single-feature ROC-AUC of every original feature (direction-free: `max(AUC, 1 - AUC)`);
2. how `URLSimilarityIndex` is distributed per class;
3. the tuned Logistic Regression with and without it (same grouped CV folds, all 50 original features).

In [ ]:
single_auc = {
    col: max(auc, 1 - auc)
    for col in original
    for auc in [roc_auc_score(y_train, X_train[col])]
}
pd.Series(single_auc, name="single-feature ROC-AUC (train)").sort_values(ascending=False).head(10).round(4)

In [ ]:
usi = X_train["URLSimilarityIndex"]
cls = y_train.map({1: "phishing", 0: "legitimate"})
print("share of rows with URLSimilarityIndex == 100, by class:")
print((usi == 100).groupby(cls).mean().round(4).to_dict())
usi.groupby(cls).describe().round(2)

In [ ]:
without = [c for c in original if c != "URLSimilarityIndex"]
usi_compare = pd.DataFrame({
    f"with URLSimilarityIndex ({len(original)} features)": evaluate.cv_evaluate(logreg.build(), X_train[original], y_train, groups=train_groups, n_jobs=2),
    f"without URLSimilarityIndex ({len(without)} features)": evaluate.cv_evaluate(logreg.build(), X_train[without], y_train, groups=train_groups, n_jobs=2),
})
usi_compare["difference (without - with)"] = usi_compare.iloc[:, 1] - usi_compare.iloc[:, 0]
usi_compare.round(5)

**How to read this for the group discussion**

- A single-feature ROC-AUC close to 1.0 means the feature alone nearly separates the classes.
  That is a warning sign of a feature that encodes how the dataset was built (for example, similarity
  to a list of known legitimate URLs) rather than something a live detector could rely on.
- If the CV metrics barely change without it, the model does not depend on it and dropping it costs little.
  If they drop clearly, the reported scores depend heavily on this one feature, and the report should say so.
- The model was retuned after the decision (section 6, `url_engineered`), so this comparison is only evidence.
- Nothing here uses the test set.

*Findings, run of 2026-11-17 (ungrouped split, historical; rerun the cells above for the grouped split):*
- `URLSimilarityIndex` has the highest single-feature ROC-AUC (0.9961). **Every legitimate training URL
  has exactly 100** (std 0), against 0.79 % of phishing URLs. It behaves like a label proxy for
  "legitimate", which suggests it was computed against the legitimate URL list used to build the dataset.
- Dropping it changes the tuned model's CV F1 from 0.99991 to 0.99965 and ROC-AUC from 1.00000 to 0.99998.
  The model barely depends on it, because **other features also nearly separate the classes on their own**
  (`LineOfCode` 0.9902, `NoOfExternalRef` 0.9880, `NoOfImage` 0.9795).
- So removing `URLSimilarityIndex` alone does not explain or remove the near-perfect scores. The
  page-content features (crawled HTML) deserve the same discussion. Group decision: dropped; page features only in the secondary `extended` set.


## 7b. Collection bias in the URL format (training split only)

Finding (2026-11-24): **every legitimate PhiUSIIL URL is a bare `https://www.<domain>` homepage** (no path, query or
fragment), while only about 1 % of phishing URLs are. The legitimate URLs were collected as homepages of listed
domains, so the URL *format* alone almost separates the classes. `data.bare_homepage_mask()` encodes the fixed
rule "not a bare https://www homepage → phishing". It learns nothing, so it can be scored on the same grouped CV
validation folds as the models. A URL-only model has learned something beyond the collection format only to the
extent that it beats this rule; `main.py` writes the same comparison on the held-out rows
(`reports/format_baseline.csv`, `reports/format_controlled.csv`).


In [ ]:

bare = data.bare_homepage_mask(df.loc[X_train.index, "URL"])
cls = y_train.map({1: "phishing", 0: "legitimate"})
print("share of bare https://www homepages by class (training rows):", bare.groupby(cls).mean().round(4).to_dict())
rule = (~bare).astype(int)
folds = evaluate.cv_folds(Xp_train, y_train, train_groups)          # identical folds to section 6
fmt_cv = pd.DataFrame({
    "format rule (no fitting)": evaluate.rule_cv_metrics(y_train, rule, folds),
    f"logreg.build() C={logreg.BEST_C:g}": cv_compare.iloc[:, 1],
})
fmt_cv.round(5)


## 8. Final test-set report

The held-out test is run **once** by `python main.py` with the frozen `build()` of every model, after all choices
above were made, and saved to `reports/official_comparison.csv` (three main models, grouped CV + test) and
`reports/format_baseline.csv` (the format rule on the same rows). This cell only displays the LR rows.
`RUN_FINAL_TEST = True` refits LR here for a cross-check; never choose anything from it.

In [ ]:
official = config.RESULTS_DIR / "official_comparison.csv"
if official.exists():
    display(pd.read_csv(official).set_index("model").T.round(5))
    display(pd.read_csv(config.RESULTS_DIR / "format_baseline.csv").T.round(5))
else:
    print("reports/official_comparison.csv not found: run `python main.py` on the full data first.")
if RUN_FINAL_TEST:
    Xp_test = df_feat.loc[X_test.index, sets[PRIMARY]]
    model = logreg.train(Xp_train, y_train)
    display(pd.Series(evaluate.evaluate(model, Xp_test, y_test)).round(5))
else:
    print("Notebook-side refit skipped (RUN_FINAL_TEST = False).")

## Experiment log

Numbers are copied from the cell outputs or the stored tuning files named in each row.

| Date | What changed | Why | CV or test | Key metrics (F1 / ROC-AUC / recall) | Commit |
|---|---|---|---|---|---|
| 2026-11-17 | Baseline: lbfgs, L2, C = 1, 50 original features | Starting point | CV (5-fold, ungrouped, full train split) | 0.99986 / 1.00000 / 0.99971 | historical, superseded |
| 2026-11-17 | Grid search C x L1/L2 (liblinear); chose L2, C = 10 | Tune regularisation (4.9) | CV (ungrouped) | 0.99991 / 1.00000 / 0.99983 | historical, superseded |
| 2026-11-17 | Tuned model without `URLSimilarityIndex` (49 features) | Leakage evidence for the group | CV (ungrouped) | 0.99965 / 0.99998 / 0.99947 | historical |
| 2026-11-24 | Group-aware split + grouped CV (`url_group_key`, `split_index`, `cv_folds`) | Near-duplicate leakage (group decision) | — | 0 URL groups across train/test (section 4b) | integration branch, uncommitted |
| 2026-11-24 | Full grouped grid, L2/lbfgs + L1/liblinear, `url_engineered` | Retune under the agreed protocol | Grouped CV | **Stopped at 36/45 fits**; L1 C = 0.1 not converged; not used for selection | `lr_full_run_INCOMPLETE_2026-11-24.log` |
| 2026-11-24 | Bounded follow-up, L2/lbfgs C 0.001–1000 (2 rounds, 50 fits, all converged) | Valid evidence within a time budget | Grouped CV | **C = 1000 (one-SE, upper edge): 0.99674 / 0.99860 / 0.99375** | `reports/tuning/lr_followup_search.csv` |
| 2026-11-24 | Format rule "not a bare https://www homepage → phishing" | Collection-bias reference | Grouped CV folds (no fitting) | section 7b; held-out: `reports/format_baseline.csv` | — |
| 2026-11-24 | Full-data `main.py`: frozen LR (C = 1000) on the held-out rows, format rule on the same rows | Final progress-evaluation numbers | CV (full train) + test | CV F1 0.9967 ± 0.0003; test F1 0.9973 / ROC-AUC 0.9985 / recall 0.9946 (FP 2, FN 108); format rule test F1 0.9948; 107 of the 108 missed phishing URLs are bare https://www homepages | `reports/official_comparison.csv`, `reports/format_controlled.csv` |